# OCR-пайплайн научных статей на русском

Универсальный пайплайн для PDF из `cyberleninka_dataset/pdfs` — извлекает:

* полный текст с восстановлением порядка чтения (1- и 2-колоночные)
* формулы (Unicode + crops, опционально LaTeX через `pix2tex`)
* таблицы (по линиям/PyMuPDF, опционально Table Transformer)
* картинки + подписи
* библиографические ссылки (русский ГОСТ-парсер)

**Архитектура**: ядро на PyMuPDF (без тяжёлых ML), плюс опциональные модули для сканов и LaTeX-OCR формул.

**Вход**: `cyberleninka_dataset/pdfs/*.pdf`  
**Выход**: `cyberleninka_dataset/ocr_out/<file>.json` + `<file>.md`


## 0. Конфиг и импорты

In [ ]:
from pathlib import Path
import re, json, statistics, math, sys, io, os
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any, Optional, Tuple
import fitz  # PyMuPDF
from PIL import Image

# === paths ===
ROOT = Path(r'c:\Users\Katherine\Documents\vkr')
PDF_DIR = ROOT / 'cyberleninka_dataset' / 'pdfs'
OUT_DIR = ROOT / 'cyberleninka_dataset' / 'ocr_out'
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'crops').mkdir(exist_ok=True)

# === optional ML toggles (set True only if libraries are installed) ===
USE_TABLE_TRANSFORMER = False  # microsoft/table-transformer-structure-recognition
USE_PIX2TEX           = False  # LaTeX-OCR for formulas
USE_SURYA_OCR         = False  # multilingual OCR for scans
USE_TESSERACT         = False  # pytesseract+rus fallback

PDFS = sorted(PDF_DIR.glob('*.pdf'))
print(f'Found {len(PDFS)} PDFs in {PDF_DIR}')


## 1. Классификация PDF (born-digital / scan / hybrid)

Решающий шаг — для 95% корпуса OCR не нужен и качество текстового слоя из PDF выше любого OCR.

In [ ]:
CYR_RE = re.compile(r'[А-Яа-яЁё]')
LAT_RE = re.compile(r'[A-Za-z]')
LETTER_RE = re.compile(r'[А-Яа-яЁёA-Za-z]')
PRINTABLE_RE = re.compile(r'[\u0020-\u007E\u00A0-\uFFFF]')  # printable Unicode

def text_decodable_ratio(text: str) -> float:
    """Share of bytes that look like real characters (not glyph indices).
    PDFs without a ToUnicode CMap return characters in the Private-Use range
    or low control bytes — those are glyph IDs, not text."""
    if not text:
        return 1.0
    nonws = [c for c in text if not c.isspace()]
    if not nonws:
        return 1.0
    bad = sum(1 for c in nonws if (ord(c) < 0x20 and c not in '\t\n\r')
                                  or 0xE000 <= ord(c) <= 0xF8FF)  # PUA
    return 1 - bad / len(nonws)

def fix_mojibake(text: str) -> str:
    """Some Russian PDFs encode text as CP1251 but PyMuPDF reads it as Latin-1.
    Symptom: lots of À-ÿ (U+00C0..U+00FF) and ~no Cyrillic.
    Fix: re-encode latin-1 -> cp1251."""
    if not text or len(text) < 3:
        return text
    nonws = [c for c in text if not c.isspace()]
    if not nonws:
        return text
    cyr = len(CYR_RE.findall(text))
    high_latin = sum(1 for c in nonws if 0x00C0 <= ord(c) <= 0x00FF)
    high_latin_ratio = high_latin / len(nonws)
    cyr_ratio = cyr / max(1, len(nonws))
    if cyr_ratio > 0.1 or high_latin_ratio < 0.3:
        return text  # not mojibake
    try:
        candidate = text.encode('latin-1', errors='ignore').decode('cp1251', errors='ignore')
    except Exception:
        return text
    cyr_after = len(CYR_RE.findall(candidate))
    return candidate if cyr_after > cyr * 3 + 2 else text

def classify_pdf(path: Path) -> Dict[str, Any]:
    """Return per-PDF stats and kind: born-digital | scan | hybrid | broken-encoding."""
    doc = fitz.open(path)
    pages_text_chars, image_dom_pages, total_img_share = [], 0, 0.0
    cyr_letters, all_letters = 0, 0
    decodable_chars = 0
    total_chars = 0
    for p in doc:
        txt = p.get_text('text') or ''
        pages_text_chars.append(len(txt))
        cyr_letters += len(CYR_RE.findall(txt))
        all_letters += len(LETTER_RE.findall(txt))
        if txt.strip():
            r = text_decodable_ratio(txt)
            decodable_chars += r * len(txt)
            total_chars += len(txt)
        page_area = (p.rect.width * p.rect.height) or 1
        img_area = 0.0
        for im in p.get_images(full=True):
            try:
                for r in p.get_image_rects(im[0]):
                    img_area += r.width * r.height
            except Exception:
                pass
        share = min(1.0, img_area / page_area)
        total_img_share += share
        if share > 0.6 and len(txt) < 300:
            image_dom_pages += 1
    n = doc.page_count or 1
    doc.close()
    cov = sum(1 for c in pages_text_chars if c >= 200) / n
    decodable = decodable_chars / max(1, total_chars)
    cyr_ratio = cyr_letters / max(1, all_letters)
    
    # broken encoding: text layer LOOKS present but it's glyph IDs / PUA — needs OCR
    text_looks_broken = decodable < 0.5 or (cov > 0.5 and all_letters < total_chars * 0.05)
    
    if cov < 0.3:
        kind = 'scan'
    elif text_looks_broken:
        kind = 'broken-encoding'
    elif image_dom_pages / n > 0.3:
        kind = 'hybrid'
    else:
        kind = 'born-digital'
    return {
        'kind': kind, 'pages': n,
        'text_layer_coverage': round(cov, 2),
        'text_decodable': round(decodable, 2),
        'cyrillic_ratio': round(cyr_ratio, 2),
        'image_dominant_pages': image_dom_pages,
        'avg_image_share': round(total_img_share / n, 2),
    }

# quick sanity check on whole corpus
from collections import Counter
kinds = Counter(classify_pdf(p)['kind'] for p in PDFS)
for k, v in kinds.items():
    print(f'  {k:18s} {v:3d}')


## 2. Layout + порядок чтения

PyMuPDF возвращает `blocks` с координатами. Для 2-колоночных страниц сортируем сначала по колонке (по медианной x-границе), потом по y. Выкидываем шапки/футеры по позиции и повторяемости.

In [ ]:
@dataclass
class Block:
    page: int
    bbox: Tuple[float, float, float, float]
    text: str
    column: int = 0
    role: str = 'body'  # body | header | footer | caption | reference | formula | table | figure

def detect_columns(blocks_xspans, page_w):
    if not blocks_xspans:
        return 1, page_w / 2
    mids = [(x0 + x1) / 2 for x0, x1 in blocks_xspans]
    rel = [m / page_w for m in mids]
    left = sum(1 for m in rel if m < 0.45)
    right = sum(1 for m in rel if m > 0.55)
    center = sum(1 for m in rel if 0.45 <= m <= 0.55)
    if left > 3 and right > 3 and (left + right) > 2 * center:
        return 2, page_w / 2
    return 1, page_w / 2

def page_blocks(page, page_idx) -> List[Block]:
    raw = page.get_text('blocks') or []
    text_blocks = [b for b in raw if len(b) >= 5 and isinstance(b[4], str) and b[4].strip()]
    xspans = [(b[0], b[2]) for b in text_blocks]
    n_cols, split_x = detect_columns(xspans, page.rect.width or 1)
    h = page.rect.height
    out = []
    for b in text_blocks:
        x0, y0, x1, y1, txt = b[0], b[1], b[2], b[3], fix_mojibake(b[4]).strip()
        col = 0
        if n_cols == 2:
            col = 0 if (x0 + x1) / 2 < split_x else 1
        # role hints (header/footer by position; refined later)
        role = 'body'
        if y1 < 0.07 * h and len(txt) < 120:
            role = 'header'
        elif y0 > 0.93 * h and len(txt) < 120:
            role = 'footer'
        out.append(Block(page=page_idx, bbox=(x0, y0, x1, y1), text=txt, column=col, role=role))
    # reading order: column, then y, then x
    out.sort(key=lambda b: (b.column, round(b.bbox[1], 1), b.bbox[0]))
    return out

# demo on first text-layer-usable PDF
_demo_pdf = next((p for p in PDFS if classify_pdf(p)['kind'] == 'born-digital'), PDFS[0])
_doc = fitz.open(_demo_pdf)
_blocks = page_blocks(_doc[0], 0)
_doc.close()
print(f'{_demo_pdf.name} page 0 blocks: {len(_blocks)}')
if _blocks:
    print(f'  first block (col={_blocks[0].column}, role={_blocks[0].role}): {_blocks[0].text[:120]!r}')


## 3. Дешифрация колонтитулов и сборка тела статьи

Шапки/подвалы выявляются как блоки, повторяющиеся по 2+ страницам в одинаковых местах. Это убирает «Вестник …», «© Иванов 2024», номера страниц и колонтитулы.

In [ ]:
import collections

def detect_recurring(all_blocks: List[Block], min_repeat=2) -> set:
    """Return text strings that repeat near top/bottom across pages — likely header/footer."""
    counter = collections.Counter()
    for b in all_blocks:
        if b.role in ('header', 'footer') or len(b.text) < 80:
            counter[b.text.strip()] += 1
    return {t for t, c in counter.items() if c >= min_repeat and len(t) > 2}

def assemble_body(all_blocks: List[Block]) -> Tuple[str, List[Block]]:
    drop = detect_recurring(all_blocks)
    kept = [b for b in all_blocks if b.text.strip() not in drop and b.role not in ('header', 'footer')]
    text = '\n\n'.join(b.text for b in kept)
    # join hyphenated line-breaks (Cyrillic): 'слово-\nпродолжение' -> 'словопродолжение'
    text = re.sub(r'([А-Яа-яЁёA-Za-z])-\n([а-яёa-z])', r'\1\2', text)
    text = re.sub(r'(?<=\S)\n(?=[а-яёa-z])', ' ', text)  # soft-wrap join
    return text, kept


## 4. Детекция формул

Две стратегии:
* **inline**: блоки с высокой плотностью math-глифов (∑∫√… + одинокие греческие буквы) → метим как `[FORMULA]…`.
* **display**: блок с центрированной короткой строкой и math-глифами → отдельный crop, опционально `pix2tex` → LaTeX.

In [ ]:
MATH_GLYPHS = set('∑∫∂∇√≈≠≤≥∞±×÷·αβγδεζηθικλμνξοπρστυφχψωΑΒΓΔΕΖΗΘΙΚΛΜΝΞΟΠΡΣΤΥΦΧΨΩ∈∉⊂⊃⊆⊇∪∩∀∃→←↔⇒⇔°⊥∥∝')
MATH_LATIN = set('xyzfghijklmnpqrt')  # italic single letters often used as variables

def math_score(text: str) -> float:
    if not text:
        return 0.0
    n = len(text)
    glyphs = sum(1 for c in text if c in MATH_GLYPHS)
    digits = sum(1 for c in text if c.isdigit())
    eq = text.count('=') + text.count('±') + text.count('≈')
    parens = text.count('(') + text.count(')')
    score = (glyphs * 3 + eq * 2 + digits * 0.3 + parens * 0.3) / max(20, n)
    return score

def tag_formulas(blocks: List[Block]) -> List[Block]:
    for b in blocks:
        if math_score(b.text) > 0.25 and len(b.text) < 400:
            b.role = 'formula'
    return blocks

# Optional LaTeX OCR for cropped formula regions
_pix2tex = None
def latex_from_image(img: 'Image.Image') -> Optional[str]:
    global _pix2tex
    if not USE_PIX2TEX:
        return None
    if _pix2tex is None:
        from pix2tex.cli import LatexOCR
        _pix2tex = LatexOCR()
    try:
        return _pix2tex(img)
    except Exception as e:
        return None

def render_block_image(page, bbox, dpi=200) -> 'Image.Image':
    rect = fitz.Rect(*bbox)
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    return Image.frombytes('RGB', (pix.width, pix.height), pix.samples)


## 5. Извлечение таблиц

Для born-digital — встроенный `page.find_tables()` PyMuPDF (использует линии и выравнивание текста). Для сканов нужен Table Transformer (флаг `USE_TABLE_TRANSFORMER`).

In [ ]:
def extract_tables_pymupdf(page) -> List[Dict[str, Any]]:
    out = []
    try:
        tabs = page.find_tables()
    except Exception:
        return out
    for t in tabs:
        try:
            data = t.extract()  # list of rows
            if not data or all(not any(c for c in row) for row in data):
                continue
            out.append({
                'bbox': list(t.bbox),
                'rows': len(data),
                'cols': max((len(r) for r in data), default=0),
                'cells': data,
            })
        except Exception:
            continue
    return out

_tt_model = _tt_proc = None
def extract_tables_tt(page_image: 'Image.Image') -> List[Dict[str, Any]]:
    """Optional Table Transformer pass — returns detected table boxes only (no cells)."""
    global _tt_model, _tt_proc
    if not USE_TABLE_TRANSFORMER:
        return []
    if _tt_model is None:
        from transformers import AutoImageProcessor, TableTransformerForObjectDetection
        import torch
        _tt_proc = AutoImageProcessor.from_pretrained('microsoft/table-transformer-detection')
        _tt_model = TableTransformerForObjectDetection.from_pretrained('microsoft/table-transformer-detection').eval()
    import torch
    inp = _tt_proc(images=page_image, return_tensors='pt')
    with torch.no_grad():
        out = _tt_model(**inp)
    target = torch.tensor([page_image.size[::-1]])
    res = _tt_proc.post_process_object_detection(out, threshold=0.7, target_sizes=target)[0]
    return [{'bbox': b.tolist(), 'score': float(s)} for b, s in zip(res['boxes'], res['scores'])]


## 6. Парсер русских ссылок (ГОСТ)

Стандарт ГОСТ Р 7.0.5: `Иванов И.И., Петров П.П. Название статьи // Журнал. — 2021. — № 3. — С. 12–25.`  
GROBID на этом формате слаб — пишем свой парсер на разделителях `//`, `—`, `№`, `С.`, `Т.`, `Вып.`

In [ ]:
REF_HEADERS_RE = re.compile(
    r'^\s*(список\s+литературы|литература|references|библиографический\s+список|использованн\w+\s+(источник|литератур))[\s.:№]*$',
    re.IGNORECASE | re.MULTILINE,
)
REF_NUM_RE = re.compile(r'^\s*(\[?\d{1,3}[\].)\]])\s+', re.MULTILINE)
DOI_RE = re.compile(r'10\.\d{4,9}/[\w\-._;()/:]+', re.IGNORECASE)
URL_RE = re.compile(r'https?://\S+|www\.\S+')
YEAR_RE = re.compile(r'\b(19[5-9]\d|20[0-3]\d)\b')
PAGES_RE = re.compile(r'(?:С\.|стр\.|pp?\.|p\.)\s*(\d+\s*[-–—]?\s*\d*)', re.IGNORECASE)
VOLISSUE_RE = re.compile(r'(?:Т\.|том|vol\.|v\.)\s*(\d+).*?(?:№|вып\.|no\.|n\.)\s*(\d+)', re.IGNORECASE)
AUTHOR_RE = re.compile(r'^([А-ЯЁ][а-яё]+(?:[-\s][А-ЯЁ][а-яё]+)?\s+[А-ЯЁ]\.\s*[А-ЯЁ]?\.?)((?:,\s*[А-ЯЁ][а-яё]+(?:[-\s][А-ЯЁ][а-яё]+)?\s+[А-ЯЁ]\.\s*[А-ЯЁ]?\.?)*)')

def find_references_block(full_text: str) -> Optional[str]:
    m = REF_HEADERS_RE.search(full_text)
    if not m:
        # fallback: numbered list of >5 entries near the end
        nums = list(REF_NUM_RE.finditer(full_text))
        if len(nums) >= 5 and nums[0].start() > len(full_text) * 0.4:
            return full_text[nums[0].start():]
        return None
    return full_text[m.end():]

def split_references(block: str) -> List[str]:
    """Split a refs block into individual entries."""
    block = re.sub(r'\s+', ' ', block).strip()
    # primary: numbered
    parts = re.split(r'(?:^|\s)(?:\[?\d{1,3}[\].)\]])\s+', block)
    parts = [p.strip() for p in parts if p.strip() and len(p.strip()) > 15]
    if len(parts) >= 3:
        return parts
    # fallback: split by author-pattern at start of sentence
    chunks = re.split(r'(?<=\.)\s+(?=[А-ЯЁ][а-яё]+\s+[А-ЯЁ]\.)', block)
    return [c for c in chunks if len(c.strip()) > 15]

def parse_one_reference(entry: str) -> Dict[str, Any]:
    e = entry.strip().rstrip('.')
    out = {'raw': entry.strip()}
    # DOI / URL
    m = DOI_RE.search(e)
    if m: out['doi'] = m.group(0).rstrip('.,;')
    m = URL_RE.search(e)
    if m: out['url'] = m.group(0).rstrip('.,;')
    # year
    yrs = YEAR_RE.findall(e)
    if yrs: out['year'] = yrs[-1]  # last year mention is usually publication year
    # pages
    m = PAGES_RE.search(e)
    if m: out['pages'] = re.sub(r'\s+', '', m.group(1))
    # volume/issue
    m = VOLISSUE_RE.search(e)
    if m:
        out['volume'] = m.group(1)
        out['issue'] = m.group(2)
    # split by '//' for journal articles
    if '//' in e:
        head, tail = e.split('//', 1)
        # head: 'Иванов И.И., Петров П.П. Название статьи'
        am = AUTHOR_RE.match(head.strip())
        if am:
            authors_str = (am.group(1) + (am.group(2) or ''))
            out['authors'] = [a.strip() for a in authors_str.split(',') if a.strip()]
            title = head.strip()[am.end():].strip(' .')
            if title: out['title'] = title
        else:
            out['title'] = head.strip(' .')
        # journal: take part before first dash/em-dash
        journal = re.split(r'\s[—–-]\s', tail.strip(' .'), maxsplit=1)[0].strip(' .')
        if journal: out['journal'] = journal
        out['type'] = 'article'
    else:
        # book / monograph
        am = AUTHOR_RE.match(e)
        if am:
            out['authors'] = [a.strip() for a in (am.group(1) + (am.group(2) or '')).split(',') if a.strip()]
            rest = e[am.end():].strip(' .')
            title = re.split(r'\s[—–-]\s', rest, maxsplit=1)[0].strip(' .')
            if title: out['title'] = title
        out['type'] = 'book'
    return out

def parse_references(full_text: str) -> Dict[str, Any]:
    block = find_references_block(full_text)
    if not block:
        return {'found': False, 'entries': [], 'raw_block': None}
    entries = split_references(block)
    parsed = [parse_one_reference(e) for e in entries]
    return {'found': True, 'count': len(parsed), 'entries': parsed,
            'raw_block_chars': len(block)}

# quick demo
_demo = '1. Иванов И.И., Петров П.П. Название статьи // Журнал экономики. — 2021. — Т. 5. — № 3. — С. 12–25. 2. Smith J. Title // Nature. — 2020. — Vol. 1. — No. 2. — pp. 3-4.'
for r in [parse_one_reference(e) for e in split_references(_demo)]:
    print(r)


## 7. OCR fallback для сканов (опциональный)

Только 4 файла из 80 — в фоновом режиме можно прогнать через Surya или Tesseract+rus. Без флага возвращает пустую строку и помечает страницу `needs_ocr=True`.

In [ ]:
_surya = None
def ocr_page_image(img: 'Image.Image') -> str:
    global _surya
    if USE_SURYA_OCR:
        if _surya is None:
            from surya.ocr import run_ocr
            from surya.model.detection.model import load_model as load_det, load_processor as load_det_p
            from surya.model.recognition.model import load_model as load_rec
            from surya.model.recognition.processor import load_processor as load_rec_p
            _surya = (load_det(), load_det_p(), load_rec(), load_rec_p())
        det_m, det_p, rec_m, rec_p = _surya
        from surya.ocr import run_ocr
        res = run_ocr([img], [['ru', 'en']], det_m, det_p, rec_m, rec_p)
        return '\n'.join(line.text for line in res[0].text_lines)
    if USE_TESSERACT:
        import pytesseract
        return pytesseract.image_to_string(img, lang='rus+eng')
    return ''  # no OCR available

def ocr_pdf_pages(path: Path, page_indices: List[int], dpi=300) -> Dict[int, str]:
    out = {}
    if not (USE_SURYA_OCR or USE_TESSERACT):
        return out
    doc = fitz.open(path)
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    for i in page_indices:
        pix = doc[i].get_pixmap(matrix=mat, alpha=False)
        img = Image.frombytes('RGB', (pix.width, pix.height), pix.samples)
        out[i] = ocr_page_image(img)
    doc.close()
    return out


## 8. Извлечение картинок и подписей

Сохраняем все встроенные изображения как PNG, привязываем подпись (`Рис. N. …` / `Fig. N. …`) по ближайшему текстовому блоку под картинкой.

In [ ]:
FIG_CAPTION_RE = re.compile(r'^(?:Рис(?:унок)?|Fig(?:ure)?|Илл\.|Ил\.)\s*\.?\s*(\d+)[.:]?\s*(.+)', re.IGNORECASE)
TAB_CAPTION_RE = re.compile(r'^(?:Таблица|Table)\s*(\d+)[.:]?\s*(.+)', re.IGNORECASE)

def extract_figures(doc: fitz.Document, blocks_by_page: Dict[int, List[Block]], save_dir: Path, stem: str) -> List[Dict[str, Any]]:
    figs = []
    for pi, page in enumerate(doc):
        for img_index, im in enumerate(page.get_images(full=True)):
            xref = im[0]
            try:
                rects = page.get_image_rects(xref)
            except Exception:
                rects = []
            if not rects:
                continue
            r = rects[0]
            # only meaningful images
            if r.width < 60 or r.height < 60:
                continue
            # extract bytes
            try:
                d = doc.extract_image(xref)
                ext = d['ext']
                fname = f'{stem}_p{pi+1}_img{img_index+1}.{ext}'
                (save_dir / fname).write_bytes(d['image'])
            except Exception:
                fname = None
            # nearest caption: block right below or above with FIG/TAB pattern
            caption, num, kind = None, None, None
            for b in blocks_by_page.get(pi, []):
                if abs(b.bbox[1] - r.y1) < 60 or abs(b.bbox[3] - r.y0) < 30:
                    m = FIG_CAPTION_RE.match(b.text)
                    if m:
                        kind, num, caption = 'figure', m.group(1), m.group(2)
                        break
                    m = TAB_CAPTION_RE.match(b.text)
                    if m:
                        kind, num, caption = 'table', m.group(1), m.group(2)
                        break
            figs.append({
                'page': pi, 'bbox': [r.x0, r.y0, r.x1, r.y1],
                'file': fname, 'kind': kind or 'figure',
                'number': num, 'caption': caption,
            })
    return figs


## 9. Сборщик пайплайна на одну статью

In [ ]:
def process_pdf(path: Path) -> Dict[str, Any]:
    meta = classify_pdf(path)
    doc = fitz.open(path)
    n_pages = doc.page_count
    use_text_layer = meta['kind'] not in ('scan', 'broken-encoding')
    
    # 1. blocks per page (text layer) — only if usable
    blocks_by_page: Dict[int, List[Block]] = {}
    all_blocks: List[Block] = []
    if use_text_layer:
        for pi, page in enumerate(doc):
            bs = page_blocks(page, pi)
            blocks_by_page[pi] = bs
            all_blocks.extend(bs)
        all_blocks = tag_formulas(all_blocks)
        for pi in blocks_by_page:
            blocks_by_page[pi] = [b for b in all_blocks if b.page == pi]
        body, _ = assemble_body(all_blocks)
    else:
        body = ''
    
    # 2. tables (only useful for usable text layer)
    tables = []
    if use_text_layer:
        for pi, page in enumerate(doc):
            for t in extract_tables_pymupdf(page):
                t['page'] = pi
                tables.append(t)
    
    # 3. figures + captions (works even for broken text — only uses image rects)
    figs_dir = OUT_DIR / 'crops'
    figures = extract_figures(doc, blocks_by_page, figs_dir, path.stem)
    
    # 4. references — only from usable text
    refs = {'found': False, 'entries': [], 'count': 0}
    if use_text_layer:
        # use already-mojibake-fixed blocks for refs parsing
        full_text_raw = '\n'.join(b.text for b in all_blocks)
        refs = parse_references(full_text_raw)
    
    # 5. OCR fallback — for scan / broken-encoding / hybrid, every page or weak pages
    needs_ocr: List[int] = []
    ocr_pages: List[Dict[str, Any]] = []
    if meta['kind'] in ('scan', 'broken-encoding'):
        needs_ocr = list(range(n_pages))
    elif meta['kind'] == 'hybrid':
        for pi, page in enumerate(doc):
            if len((page.get_text('text') or '')) < 200:
                needs_ocr.append(pi)
    if needs_ocr and (USE_SURYA_OCR or USE_TESSERACT):
        ocr_results = ocr_pdf_pages(path, needs_ocr)
        for pi, txt in ocr_results.items():
            ocr_pages.append({'page': pi, 'text': txt})
        # if body empty (broken/scan), assemble from OCR
        if not body:
            body = '\n\n'.join(p['text'] for p in sorted(ocr_pages, key=lambda x: x['page']))
            refs = parse_references(body)
    
    doc.close()
    
    formulas = [b for b in all_blocks if b.role == 'formula']
    
    return {
        'file': path.name,
        'meta': meta,
        'body_text': body,
        'body_chars': len(body),
        'blocks': [{'page': b.page, 'bbox': list(b.bbox), 'text': b.text,
                    'role': b.role, 'column': b.column} for b in all_blocks],
        'formulas': [{'page': b.page, 'bbox': list(b.bbox), 'text': b.text} for b in formulas],
        'tables': tables,
        'figures': figures,
        'references': refs,
        'needs_ocr_pages': needs_ocr,
        'ocr_pages': ocr_pages,
        'ocr_available': bool(USE_SURYA_OCR or USE_TESSERACT),
    }


## 10. Markdown-рендер для одной статьи

In [ ]:
def render_markdown(art: Dict[str, Any]) -> str:
    md_lines = [f"# {art['file']}\n"]
    m = art['meta']
    md_lines.append(f"- **kind**: {m['kind']}  • pages: {m['pages']}  • text-coverage: {m['text_layer_coverage']}  • cyrillic: {m['cyrillic_ratio']}\n")
    md_lines.append('\n## Текст\n')
    md_lines.append(art['body_text'])
    if art['formulas']:
        md_lines.append('\n\n## Формулы\n')
        for f in art['formulas']:
            md_lines.append(f"- (стр. {f['page']+1}) `{f['text']}`\n")
    if art['tables']:
        md_lines.append('\n## Таблицы\n')
        for ti, t in enumerate(art['tables'], 1):
            md_lines.append(f"### Таблица {ti} (стр. {t['page']+1}, {t['rows']}x{t['cols']})\n")
            for row in t['cells'][:20]:
                md_lines.append('| ' + ' | '.join((c or '').replace('\n', ' ') for c in row) + ' |\n')
            md_lines.append('\n')
    if art['figures']:
        md_lines.append('\n## Рисунки\n')
        for f in art['figures']:
            cap = f.get('caption') or '(без подписи)'
            md_lines.append(f"- {f['kind']} стр. {f['page']+1} — {cap}\n")
    refs = art['references']
    if refs.get('found'):
        md_lines.append(f"\n## Литература ({refs['count']} записей)\n")
        for i, r in enumerate(refs['entries'], 1):
            md_lines.append(f"{i}. {r.get('raw','')[:300]}\n")
    return ''.join(md_lines)


## 11. Прогон на всём корпусе

In [ ]:
import time
results = []
errors = []
t0 = time.time()
for i, p in enumerate(PDFS, 1):
    try:
        art = process_pdf(p)
        # write outputs
        json_out = OUT_DIR / f'{p.stem}.json'
        json_out.write_text(json.dumps(art, ensure_ascii=False, indent=2), encoding='utf-8')
        md_out = OUT_DIR / f'{p.stem}.md'
        md_out.write_text(render_markdown(art), encoding='utf-8')
        results.append(art)
    except Exception as e:
        errors.append((p.name, str(e)))
        print(f'  ERROR {p.name}: {e}')
    if i % 10 == 0:
        print(f'  {i}/{len(PDFS)} done  ({time.time()-t0:.1f}s)')
print(f'\nDone: {len(results)} ok, {len(errors)} errors, {time.time()-t0:.1f}s total')


## 12. Отчёт по качеству

Метрики, которые считаем без размеченного gold set:
* `text_layer_coverage` (доля страниц с текстом)
* `body_chars / page` (sanity)
* `references_found_rate` и средняя `n_entries`
* `formulas_per_pdf`, `tables_per_pdf`, `figures_per_pdf`
* по доменам — для оценки робастности.

In [ ]:
def report(results: List[Dict[str, Any]]):
    n = len(results)
    print(f'=== Pipeline report over {n} PDFs ===\n')
    kinds = collections.Counter(r['meta']['kind'] for r in results)
    for k, v in kinds.items():
        print(f'  kind={k:12s}  {v}  ({v/n:.0%})')
    cov = statistics.mean(r['meta']['text_layer_coverage'] for r in results)
    print(f'\n  mean text_layer_coverage : {cov:.2f}')
    print(f'  mean body_chars/page     : {statistics.mean(r["body_chars"]/max(1,r["meta"]["pages"]) for r in results):.0f}')
    rfound = sum(1 for r in results if r['references']['found'])
    print(f'  references found         : {rfound}/{n} ({rfound/n:.0%})')
    if rfound:
        avg_ref = statistics.mean(r['references']['count'] for r in results if r['references']['found'])
        print(f'  mean entries per refs    : {avg_ref:.1f}')
    print(f'  formulas total           : {sum(len(r["formulas"]) for r in results)}')
    print(f'  tables total             : {sum(len(r["tables"]) for r in results)}')
    print(f'  figures total            : {sum(len(r["figures"]) for r in results)}')
    
    # by domain
    print('\n=== by domain ===')
    by_dom = {}
    for r in results:
        dom = r['file'].split('_')[0]
        by_dom.setdefault(dom, []).append(r)
    for dom, rs in sorted(by_dom.items()):
        ref_rate = sum(1 for r in rs if r['references']['found']) / len(rs)
        body_chars = statistics.mean(r['body_chars'] / max(1, r['meta']['pages']) for r in rs)
        formulas = sum(len(r['formulas']) for r in rs)
        tables = sum(len(r['tables']) for r in rs)
        print(f'  {dom:8s}  n={len(rs):2d}  body_chars/p={body_chars:5.0f}  '
              f'refs_found={ref_rate:4.0%}  formulas={formulas:3d}  tables={tables:3d}')
    
    if errors:
        print(f'\n=== ERRORS ===')
        for name, e in errors:
            print(f'  {name}: {e}')

report(results)


## 13. Включение опциональных модулей

По умолчанию пайплайн работает на чистом PyMuPDF. Чтобы включить тяжёлые модели:

```bash
# для 4 сканов:
pip install surya-ocr   # либо: pip install pytesseract  (+ системный tesseract-ocr-rus)

# для LaTeX-OCR формул:
pip install pix2tex

# для Table Transformer (детекция таблиц на сканах):
# transformers и torch уже стоят
```

Затем в ячейке 0 поменять флаги `USE_SURYA_OCR=True` / `USE_PIX2TEX=True` / `USE_TABLE_TRANSFORMER=True` и перезапустить ноутбук.

## 14. Метрики качества с разметкой

Когда появится gold-набор (10-20 размеченных PDF):
* **CER/WER** на полном тексте (через `jiwer`)
* **TEDS** на таблицах (через `apted`)
* **Per-field F1** на ссылках: author, title, year, journal, DOI
* **ExpRate / BLEU** на формулах в LaTeX
* кросс-стратовая дисперсия (born-digital vs scan vs 2-col vs formula-heavy) — проверка «универсальности».